
# GRIDPILOT AI — Member 2 — Task 1
## OpenSTEF Liander 2024 Dataset Inspection

**Purpose:** This notebook performs **Task 1 only** for Member 2.

It does NOT train LightGBM/XGBoost, build the forecasting service, or perform production cleaning.  
The goal is to inspect the actual OpenSTEF Liander dataset and produce the evidence needed for:

`docs/data/openstef-demand.md`

### What this notebook investigates
1. Dataset loading
2. File/shape/schema inspection
3. Column names and data types
4. Timestamp identification and parsing
5. Sampling frequency
6. Timezone
7. Demand/load candidate fields
8. Zone/feeder/location fields
9. Weather fields
10. Solar-related fields
11. Wind-related fields
12. Missing values
13. Duplicate records
14. Row grain
15. Demand statistics
16. Possible leakage fields
17. Candidate targets for T+15/T+30/T+60
18. Automated inspection report
19. Export of an inspection report and column profile

> **Important:** The notebook intentionally does not invent OpenSTEF column names. It first shows the actual columns and lets you select the relevant fields.



## 0. Install / import libraries

This notebook is designed for Google Colab.


In [1]:

!pip -q install pandas numpy matplotlib openpyxl pyarrow

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Libraries loaded successfully.



## 1. Upload or locate the OpenSTEF Liander dataset

### Option A — Upload from your computer

Run the next cell and choose the OpenSTEF file.

Supported common formats:
- CSV
- Parquet
- Excel
- JSON

### Option B — Use Google Drive

If the dataset is already in Drive, mount Drive and enter the file path in the configuration cell below.


In [4]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
      --------------------------------------- 10.2/559.1 kB ? eta -:--:--
      --------------------------------------- 10.2/559.1 kB ? eta -:--:--
     -- ---------------------------------- 41.0/559.1 kB 245.8 kB/s eta 0:00:03
     ---- -------------------------------- 61.4/559.1 kB 297.7 kB/s eta 0:00:02
     ------ ------------------------------ 92.2/559.1 kB 374.1 kB/s eta 0:00:02
     --------- -------------------------- 143.4/559.1 kB 500.5 kB/s eta 0:00:01
     ----------- ------------------------ 174.1/559.1 kB 525.1 kB/s eta 0:00:01
     ------------- ---------------------- 204.8/559.1 kB 541.9 kB/s eta 0:00:01
     --------------- -------------------- 245.8/559.1 kB 602.4 kB/s eta 0:00:01
     ---------------- ------------------- 256.0/559.1 kB 582.4 kB/s eta 0:00:01
     ----------------- ------------------ 276.5/559.1 kB 567.


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from datasets import load_dataset

dataset = load_dataset("OpenSTEF/liander2024-energy-forecasting-benchmark")


ImportError: The pyarrow installation is not built with support for 'dataset' (DLL load failed while importing _fs: An Application Control policy has blocked this file.)

## 2. Load the dataset

In [ ]:

def load_dataset(path):
    path = str(path)
    suffix = Path(path).suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)

    if suffix in [".parquet", ".pq"]:
        return pd.read_parquet(path)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)

    raise ValueError(
        f"Unsupported file format: {suffix}. "
        "Use CSV, Parquet, Excel, or JSON."
    )

df = load_dataset(DATA_PATH)

print("Dataset loaded successfully.")
print("Path:", DATA_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))


NameError: name 'DATA_PATH' is not defined

## 3. Basic dataset overview

In [ ]:

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:3}. {col}")


## 4. Column profile

In [ ]:

column_profile = pd.DataFrame({
    "column": df.columns.astype(str),
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null": [df[c].notna().sum() for c in df.columns],
    "missing": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [df[c].isna().mean() * 100 for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
})

display(column_profile)



## 5. Automatically find candidate columns

This section only **suggests** columns based on their names.

It does not decide that a field is demand, weather, zone, etc.

You must verify the meaning from the actual dataset/source documentation.


In [ ]:

def suggest_columns(columns, keywords):
    results = []
    for col in columns:
        text = str(col).lower()
        matched = [k for k in keywords if k.lower() in text]
        if matched:
            results.append((col, matched))
    return results

keyword_groups = {
    "timestamp": ["timestamp", "datetime", "date", "time", "utc"],
    "demand_load": ["demand", "load", "consumption", "electricity"],
    "zone": ["zone", "region", "area", "location", "grid"],
    "feeder": ["feeder"],
    "weather": [
        "temperature", "temp", "humidity", "cloud",
        "wind", "weather", "pressure", "radiation"
    ],
    "solar": ["solar", "pv", "photovoltaic", "irradiance"],
    "wind": ["wind", "windpower", "wind_generation"],
    "forecast": ["forecast", "prediction", "predicted"],
}

for group, keywords in keyword_groups.items():
    print(f"\n### {group.upper()}")
    matches = suggest_columns(df.columns, keywords)
    if matches:
        for col, matched in matches:
            print(f"- {col}  <-- matched: {matched}")
    else:
        print("No name-based candidate found.")



## 6. Select the actual important fields

After reviewing the column list above, enter the **actual column names** below.

Do not use example names unless they really exist in your dataset.

If a field does not exist, leave it as `None`.

The demand column is the most important selection.


In [ ]:

# ====== REQUIRED MANUAL CONFIGURATION ======
# Replace the values with the ACTUAL column names from your dataset.

TIMESTAMP_COL = None       # Example only: "timestamp"
DEMAND_COL = None          # Example only: "actual_load"
ZONE_COL = None            # Example only: "zone_id"
FEEDER_COL = None          # Example only: "feeder_id"

# Optional fields. Leave None if unavailable.
TEMPERATURE_COL = None
HUMIDITY_COL = None
CLOUD_COVER_COL = None
WIND_SPEED_COL = None
SOLAR_RADIATION_COL = None

# ===========================================

selected = {
    "timestamp": TIMESTAMP_COL,
    "demand": DEMAND_COL,
    "zone": ZONE_COL,
    "feeder": FEEDER_COL,
    "temperature": TEMPERATURE_COL,
    "humidity": HUMIDITY_COL,
    "cloud_cover": CLOUD_COVER_COL,
    "wind_speed": WIND_SPEED_COL,
    "solar_radiation": SOLAR_RADIATION_COL,
}

print("Selected fields:")
for k, v in selected.items():
    print(f"{k:20} -> {v}")


In [ ]:

def validate_selected_columns(df, selections):
    errors = []
    for logical_name, col in selections.items():
        if col is not None and col not in df.columns:
            errors.append(
                f"{logical_name}: '{col}' does not exist in the dataset."
            )
    if errors:
        print("FIELD SELECTION ERRORS:")
        for e in errors:
            print(" -", e)
        raise ValueError("Fix the selected column names above.")
    print("All selected columns exist.")

validate_selected_columns(df, selected)


## 7. Inspect the timestamp

In [ ]:

if TIMESTAMP_COL is None:
    print("TIMESTAMP_COL is None. Select the actual timestamp field first.")
else:
    print("Raw timestamp examples:")
    display(df[TIMESTAMP_COL].head(10))

    timestamp_parsed = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")

    print("\nParsed timestamp dtype:", timestamp_parsed.dtype)
    print("Invalid/unparseable timestamps:", timestamp_parsed.isna().sum())

    if timestamp_parsed.notna().any():
        print("Minimum timestamp:", timestamp_parsed.min())
        print("Maximum timestamp:", timestamp_parsed.max())


## 8. Determine sampling frequency

In [ ]:

def frequency_analysis(data, timestamp_col, group_col=None):
    temp = data[[timestamp_col] + ([group_col] if group_col else [])].copy()
    temp["_ts"] = pd.to_datetime(temp[timestamp_col], errors="coerce")
    temp = temp.dropna(subset=["_ts"])

    if group_col:
        temp = temp.sort_values([group_col, "_ts"])
        temp["_delta"] = temp.groupby(group_col)["_ts"].diff()
    else:
        temp = temp.sort_values("_ts")
        temp["_delta"] = temp["_ts"].diff()

    counts = temp["_delta"].value_counts()
    return counts

if TIMESTAMP_COL is None:
    print("Select TIMESTAMP_COL first.")
else:
    freq_counts = frequency_analysis(
        df,
        TIMESTAMP_COL,
        ZONE_COL if ZONE_COL is not None else None
    )

    print("Most common time intervals:")
    display(freq_counts.head(15).rename("count").to_frame())

    if len(freq_counts):
        print("Most common interval:", freq_counts.index[0])



### Frequency interpretation

For your later forecasting work, you need to know whether the data is:

- 15-minute
- 30-minute
- hourly
- another interval
- irregular

Do not assume 15 minutes merely because the final forecast horizons are 15/30/60 minutes.


## 9. Determine timezone

In [ ]:

if TIMESTAMP_COL is not None:
    ts = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")

    print("Timestamp dtype:", ts.dtype)

    if getattr(ts.dt, "tz", None) is not None:
        print("Timezone detected:", ts.dt.tz)
    else:
        print(
            "Timezone is not encoded in the parsed values "
            "(timezone-naive timestamps)."
        )

    print("\nIMPORTANT:")
    print("Verify the official dataset documentation for the intended timezone.")
    print("Do not guess the timezone from the machine/Colab timezone.")


## 10. Inspect the demand/load field

In [ ]:

if DEMAND_COL is None:
    print("Select DEMAND_COL first.")
else:
    print("Demand field:", DEMAND_COL)
    print("Data type:", df[DEMAND_COL].dtype)
    print("Missing:", df[DEMAND_COL].isna().sum())
    print("Unique:", df[DEMAND_COL].nunique())

    numeric_demand = pd.to_numeric(df[DEMAND_COL], errors="coerce")

    print("\nDemand statistics:")
    display(numeric_demand.describe().to_frame("value"))

    print("\nSample demand values:")
    display(df[[DEMAND_COL]].head(20))



### Demand unit

This notebook cannot safely infer whether demand is MW, kW, GW, etc. from the numbers alone.

Verify the unit from the actual OpenSTEF dataset documentation/metadata and record it in:

`docs/data/openstef-demand.md`

Do not guess.


## 11. Inspect zone / feeder / location information

In [ ]:

for logical_name, col in [
    ("ZONE", ZONE_COL),
    ("FEEDER", FEEDER_COL),
]:
    print(f"\n===== {logical_name} =====")

    if col is None:
        print("Not selected / not available.")
        continue

    print("Column:", col)
    print("Unique values:", df[col].nunique(dropna=True))
    print("Missing:", df[col].isna().sum())

    values = df[col].dropna().unique()
    print("First values:", values[:30])



## 12. Determine the row grain

The row grain means:

> What exactly does one row represent?

Possible examples:

`timestamp`

or:

`timestamp + zone`

or:

`timestamp + feeder`

or another combination.

This matters when deciding what counts as a duplicate.


In [ ]:

candidate_keys = []

if TIMESTAMP_COL is not None:
    candidate_keys.append(["timestamp"])

if TIMESTAMP_COL is not None and ZONE_COL is not None:
    candidate_keys.append(["timestamp", "zone"])

if TIMESTAMP_COL is not None and FEEDER_COL is not None:
    candidate_keys.append(["timestamp", "feeder"])

for key_name in candidate_keys:
    print("\nCandidate key:", " + ".join(key_name))

    mapping = {
        "timestamp": TIMESTAMP_COL,
        "zone": ZONE_COL,
        "feeder": FEEDER_COL,
    }

    cols = [mapping[x] for x in key_name]
    duplicate_count = df.duplicated(subset=cols).sum()

    print("Duplicate rows using this key:", duplicate_count)


## 13. Missing-value analysis

In [ ]:

missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [df[c].isna().mean() * 100 for c in df.columns],
}).sort_values("missing_pct", ascending=False)

display(missing_report)


## 14. Duplicate analysis

In [ ]:

print("Duplicate rows across ALL columns:", df.duplicated().sum())

if TIMESTAMP_COL is not None:
    print(
        "Duplicate timestamps:",
        df.duplicated(subset=[TIMESTAMP_COL]).sum()
    )

if TIMESTAMP_COL is not None and ZONE_COL is not None:
    print(
        "Duplicate timestamp + zone:",
        df.duplicated(subset=[TIMESTAMP_COL, ZONE_COL]).sum()
    )

if TIMESTAMP_COL is not None and FEEDER_COL is not None:
    print(
        "Duplicate timestamp + feeder:",
        df.duplicated(subset=[TIMESTAMP_COL, FEEDER_COL]).sum()
    )



## 15. Check demand values for suspicious ranges

This is an inspection step, NOT the final cleaning rule.

Negative values, zeros, extreme values, etc. must be interpreted according to the actual dataset semantics.


In [ ]:

if DEMAND_COL is not None:
    demand_numeric = pd.to_numeric(df[DEMAND_COL], errors="coerce")

    print("Negative demand count:", (demand_numeric < 0).sum())
    print("Zero demand count:", (demand_numeric == 0).sum())
    print("Positive demand count:", (demand_numeric > 0).sum())

    print("\nFive smallest values:")
    display(demand_numeric.nsmallest(5).to_frame("demand"))

    print("\nFive largest values:")
    display(demand_numeric.nlargest(5).to_frame("demand"))


## 16. Weather / solar / wind availability

In [ ]:

weather_fields = {
    "temperature": TEMPERATURE_COL,
    "humidity": HUMIDITY_COL,
    "cloud_cover": CLOUD_COVER_COL,
    "wind_speed": WIND_SPEED_COL,
    "solar_radiation": SOLAR_RADIATION_COL,
}

for logical_name, col in weather_fields.items():
    print(f"\n===== {logical_name.upper()} =====")

    if col is None:
        print("Not selected / not available.")
        continue

    print("Column:", col)
    print("dtype:", df[col].dtype)
    print("missing:", df[col].isna().sum())
    print("unique:", df[col].nunique(dropna=True))

    display(df[col].describe().to_frame("value"))



### Broader automatic search for weather/solar/wind fields

Review the candidate names and manually verify their meaning.


In [ ]:

for group in ["weather", "solar", "wind"]:
    print(f"\n===== POSSIBLE {group.upper()} FIELDS =====")
    matches = suggest_columns(df.columns, keyword_groups[group])

    if not matches:
        print("None found by name.")
    else:
        for col, matched in matches:
            print(f"- {col} <- {matched}")


## 17. Possible leakage-field investigation

In [ ]:

# Automatically flag columns whose names may indicate future/predicted information.
leakage_keywords = [
    "future",
    "ahead",
    "lead",
    "target",
    "actual_future",
    "future_demand",
    "future_load",
    "forecast",
    "predicted",
    "prediction",
]

possible_leakage = []

for col in df.columns:
    text = str(col).lower()
    matched = [k for k in leakage_keywords if k in text]
    if matched:
        possible_leakage.append({
            "column": col,
            "matched_keywords": ", ".join(matched),
            "dtype": str(df[col].dtype),
            "unique_values": df[col].nunique(dropna=True),
            "missing_pct": df[col].isna().mean() * 100,
        })

leakage_report = pd.DataFrame(possible_leakage)

if leakage_report.empty:
    print("No obvious leakage candidates were found by column name.")
else:
    display(leakage_report)

print("\nIMPORTANT:")
print("A column name alone does NOT prove leakage.")
print("Verify whether each field would actually be available at prediction time.")



## 18. Candidate forecast targets: T+15 / T+30 / T+60

This section does NOT train a model.

It only checks whether future target rows can theoretically be constructed from the timestamp series.

For example, if the data frequency is 15 minutes:

- T+15 = one interval ahead
- T+30 = two intervals ahead
- T+60 = four intervals ahead

The exact interval count must be based on the actual discovered frequency.


In [ ]:

def infer_common_minutes(data, timestamp_col, group_col=None):
    counts = frequency_analysis(data, timestamp_col, group_col)

    if counts.empty:
        return None

    delta = counts.index[0]

    if pd.isna(delta):
        return None

    return delta.total_seconds() / 60

if TIMESTAMP_COL is not None:
    common_minutes = infer_common_minutes(
        df,
        TIMESTAMP_COL,
        ZONE_COL if ZONE_COL is not None else None
    )

    print("Most common interval in minutes:", common_minutes)

    if common_minutes:
        for horizon in [15, 30, 60]:
            intervals = horizon / common_minutes
            print(
                f"T+{horizon}: {intervals} dataset intervals ahead"
                if intervals.is_integer()
                else
                f"T+{horizon}: NOT an exact integer number of "
                f"dataset intervals ({intervals})"
            )



## 19. Check timestamp coverage and gaps

This helps determine whether the dataset is continuous enough for later forecasting work.

This is still inspection only. No missing timestamps are filled here.


In [ ]:

if TIMESTAMP_COL is not None:
    temp = df[[TIMESTAMP_COL] + ([ZONE_COL] if ZONE_COL else [])].copy()
    temp["_ts"] = pd.to_datetime(temp[TIMESTAMP_COL], errors="coerce")
    temp = temp.dropna(subset=["_ts"])

    if ZONE_COL is not None:
        gap_summary = []

        for zone, group in temp.groupby(ZONE_COL):
            s = group["_ts"].drop_duplicates().sort_values()
            gaps = s.diff().dropna()

            if len(gaps):
                gap_summary.append({
                    "zone": zone,
                    "rows": len(group),
                    "min_timestamp": s.min(),
                    "max_timestamp": s.max(),
                    "median_interval": gaps.median(),
                    "largest_interval": gaps.max(),
                    "number_of_intervals": len(gaps),
                })

        if gap_summary:
            gap_df = pd.DataFrame(gap_summary)
            display(gap_df.head(50))
    else:
        s = temp["_ts"].drop_duplicates().sort_values()
        gaps = s.diff().dropna()

        print("Unique timestamps:", len(s))
        print("Median interval:", gaps.median())
        print("Largest interval:", gaps.max())


## 20. Optional demand visualization

In [ ]:

if TIMESTAMP_COL is not None and DEMAND_COL is not None:
    plot_df = df[[TIMESTAMP_COL, DEMAND_COL]].copy()
    plot_df["timestamp"] = pd.to_datetime(
        plot_df[TIMESTAMP_COL],
        errors="coerce"
    )
    plot_df["demand"] = pd.to_numeric(
        plot_df[DEMAND_COL],
        errors="coerce"
    )
    plot_df = plot_df.dropna().sort_values("timestamp")

    if len(plot_df) > 0:
        plt.figure(figsize=(14, 5))
        plt.plot(plot_df["timestamp"], plot_df["demand"])
        plt.xlabel("Time")
        plt.ylabel("Demand")
        plt.title("OpenSTEF Demand — Raw Inspection")
        plt.grid(True)
        plt.show()
    else:
        print("No valid timestamp+demand rows available for plotting.")



## 21. Build an automated Task 1 summary

The following cell creates a machine-readable summary from the inspection results.

You should manually review it before committing the final documentation.


In [ ]:

summary = {
    "dataset_file": str(DATA_PATH),
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "column_names": [str(c) for c in df.columns],
    "selected_fields": selected,
}

if TIMESTAMP_COL is not None:
    ts = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")
    summary["timestamp"] = {
        "invalid_count": int(ts.isna().sum()),
        "min": str(ts.min()) if ts.notna().any() else None,
        "max": str(ts.max()) if ts.notna().any() else None,
        "dtype": str(ts.dtype),
        "timezone": str(ts.dt.tz) if ts.dt.tz is not None else "timezone-naive",
    }

if DEMAND_COL is not None:
    d = pd.to_numeric(df[DEMAND_COL], errors="coerce")
    summary["demand"] = {
        "missing_count": int(d.isna().sum()),
        "min": float(d.min()) if d.notna().any() else None,
        "max": float(d.max()) if d.notna().any() else None,
        "mean": float(d.mean()) if d.notna().any() else None,
        "median": float(d.median()) if d.notna().any() else None,
        "negative_count": int((d < 0).sum()),
        "zero_count": int((d == 0).sum()),
    }

summary["duplicates"] = {
    "all_columns": int(df.duplicated().sum()),
}

if TIMESTAMP_COL is not None:
    summary["duplicates"]["timestamp"] = int(
        df.duplicated(subset=[TIMESTAMP_COL]).sum()
    )

if TIMESTAMP_COL is not None and ZONE_COL is not None:
    summary["duplicates"]["timestamp_zone"] = int(
        df.duplicated(subset=[TIMESTAMP_COL, ZONE_COL]).sum()
    )

summary["missing_values"] = {
    str(c): {
        "count": int(df[c].isna().sum()),
        "percentage": float(df[c].isna().mean() * 100)
    }
    for c in df.columns
}

summary["possible_leakage_columns"] = (
    leakage_report.to_dict(orient="records")
    if not leakage_report.empty else []
)

display(pd.DataFrame([
    ["Dataset rows", summary["rows"]],
    ["Dataset columns", summary["columns"]],
    ["Timestamp field", TIMESTAMP_COL],
    ["Demand field", DEMAND_COL],
    ["Zone field", ZONE_COL],
    ["Feeder field", FEEDER_COL],
    ["Possible leakage fields", len(summary["possible_leakage_columns"])],
], columns=["Item", "Value"]))



## 22. Export inspection results

These files are useful for your project documentation.

They are **inspection artifacts**, not production data-cleaning outputs.


In [ ]:

OUTPUT_DIR = Path("/content/gridpilot_task1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save column profile
column_profile.to_csv(
    OUTPUT_DIR / "openstef_column_profile.csv",
    index=False
)

# Save missing-value report
missing_report.to_csv(
    OUTPUT_DIR / "openstef_missing_values.csv",
    index=False
)

# Save possible leakage report
leakage_report.to_csv(
    OUTPUT_DIR / "openstef_possible_leakage.csv",
    index=False
)

# Save JSON summary
with open(OUTPUT_DIR / "openstef_task1_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)



# 23. FINAL TASK 1 CHECKLIST

Before marking Task 1 complete, verify:

- [ ] Actual OpenSTEF Liander dataset identified
- [ ] Dataset source recorded
- [ ] Date coverage identified
- [ ] Number of rows/columns recorded
- [ ] Exact timestamp field identified
- [ ] Timestamp parsing checked
- [ ] Sampling frequency determined
- [ ] Timezone verified from source documentation
- [ ] Exact demand/load field identified
- [ ] Demand unit verified from source documentation
- [ ] Zone/feeders/location fields identified
- [ ] Row grain determined
- [ ] Weather fields identified
- [ ] Solar fields identified
- [ ] Wind fields identified
- [ ] Missing values measured
- [ ] Duplicate behavior investigated
- [ ] Demand statistics recorded
- [ ] Possible leakage fields investigated
- [ ] Candidate T+15/T+30/T+60 targets documented
- [ ] No production cleaning/model code written
- [ ] `docs/data/openstef-demand.md` prepared using actual findings

## Next task

After this notebook is complete, **Task 2** can use these findings to implement:

`Data Ingestion → Validation → Cleaning → Canonical Dataset`

Do not jump to LightGBM/XGBoost until the data contract and cleaning behavior are established.
